# 01. EDA & Preprocessing

이 노트북은 `src/preprocessing.py`, `src/train.py`, `src/inference.py`, `src/monitor.py`, `tests/test_pipeline.py`를 기준으로 데이터 로드, 탐색적 분석, 전처리 검증을 재현합니다.

## 목표
- 원본 UCI heart disease 파일들을 병합하는 방식을 확인합니다.
- `load_data()`가 수행하는 중복 제거, 컬럼 제거, 타깃 이진화를 재현합니다.
- 연속형/범주형 변수의 분포와 결측 현황을 확인합니다.
- 연속형 변수별 `IQR` 기준 이상치 개수를 계산합니다.
- `build_pipeline()` 출력이 결정론적이며 결측이 없음을 검증합니다.
- 이후 `train.py`, `inference.py`, `monitor.py`에서 사용하는 입력 형식을 준비합니다.


In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display
from pandas.testing import assert_frame_equal, assert_series_equal
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_theme(style="whitegrid", context="notebook")

BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent
elif not (BASE_DIR / "src").exists() and (BASE_DIR.parent / "src").exists():
    BASE_DIR = BASE_DIR.parent

sys.path.append(str(BASE_DIR / "src"))

from preprocessing import (
    RAW_FILES,
    COLUMN_NAMES,
    DROP_COLS,
    TARGET_COL,
    FEATURE_COLS,
    CONTINUOUS_COLS,
    CATEGORICAL_COLS,
    VALID_RANGES,
    load_data,
    build_pipeline,
    validate_input_ranges,
)

DATA_DIR = BASE_DIR / "data"
RANDOM_STATE = 42
TEST_SIZE = 0.2

print(f"BASE_DIR: {BASE_DIR}")
print(f"DATA_DIR: {DATA_DIR}")


## 1. 원본 데이터 로드

`preprocessing.py`는 네 개의 processed 데이터 파일을 읽어서 하나의 데이터프레임으로 합칩니다. 먼저 원본 병합 상태를 확인합니다.


In [ ]:
def load_raw_dataframe(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    frames = []
    for source_name, filename in RAW_FILES.items():
        file_path = data_dir / filename
        df_part = pd.read_csv(file_path, header=None, names=COLUMN_NAMES, na_values="?")
        df_part["source"] = source_name
        frames.append(df_part)
    return pd.concat(frames, ignore_index=True)

file_summary = pd.DataFrame(
    [
        {
            "source": source_name,
            "file_name": filename,
            "exists": (DATA_DIR / filename).exists(),
            "file_size_bytes": (DATA_DIR / filename).stat().st_size if (DATA_DIR / filename).exists() else np.nan,
        }
        for source_name, filename in RAW_FILES.items()
    ]
)

raw_df = load_raw_dataframe(DATA_DIR)

display(file_summary)
print(f"Raw dataframe shape: {raw_df.shape}")
raw_df.head()


In [ ]:
raw_profile = pd.DataFrame(
    {
        "dtype": raw_df.dtypes.astype(str),
        "missing_count": raw_df.isna().sum(),
        "missing_ratio_pct": (raw_df.isna().mean() * 100).round(2),
    }
)

display(raw_profile.sort_values(["missing_count", "dtype"], ascending=[False, True]))
print(f"Duplicated rows in raw dataframe: {raw_df.duplicated().sum()}")
print("\nRaw target distribution before binarization")
display(raw_df[TARGET_COL].value_counts(dropna=False).sort_index().to_frame("count"))


In [ ]:
binary_target_for_eda = (raw_df[TARGET_COL] > 0).astype(int)
source_target_ratio = pd.crosstab(raw_df["source"], binary_target_for_eda, normalize="index")
source_target_ratio = source_target_ratio.rename(columns={0: "No Disease", 1: "Disease"})

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.countplot(data=raw_df, x="source", order=raw_df["source"].value_counts().index, color="steelblue", ax=axes[0])
axes[0].set_title("Records per Source")
axes[0].set_xlabel("Source")
axes[0].set_ylabel("Count")

source_target_ratio.plot(kind="bar", stacked=True, ax=axes[1], color=["#9ecae1", "#fc9272"])
axes[1].set_title("Binary Target Ratio by Source")
axes[1].set_xlabel("Source")
axes[1].set_ylabel("Ratio")
axes[1].legend(title="Target", loc="upper right")

plt.tight_layout()
plt.show()


## 2. `preprocessing.py` 로직 재현

이 단계에서는 실제 함수와 동일하게 중복 제거, 불필요 컬럼 제거, 타깃 이진화를 수행하고 `load_data()` 결과와 일치하는지 확인합니다.


In [ ]:
eda_df = raw_df.drop_duplicates().copy()
cols_to_drop = [col for col in DROP_COLS if col in eda_df.columns]

clean_df = eda_df.drop(columns=cols_to_drop).copy()
clean_df[TARGET_COL] = (clean_df[TARGET_COL] > 0).astype(int)
clean_df = clean_df.dropna(subset=[TARGET_COL])

X, y = load_data(DATA_DIR)
project_df = X.copy()
project_df[TARGET_COL] = y.values

assert list(X.columns) == FEATURE_COLS
assert_frame_equal(clean_df[FEATURE_COLS].reset_index(drop=True), X.reset_index(drop=True))
assert_series_equal(clean_df[TARGET_COL].reset_index(drop=True), y.reset_index(drop=True), check_names=False)

print(f"Columns dropped: {cols_to_drop}")
print(f"Modeling dataframe shape: {project_df.shape}")
project_df.head()


In [ ]:
display(project_df[CONTINUOUS_COLS].describe().T)

class_ratio = (
    project_df[TARGET_COL]
    .value_counts(normalize=True)
    .sort_index()
    .rename(index={0: "No Disease", 1: "Disease"})
    .mul(100)
    .round(2)
    .to_frame("ratio_pct")
)
missing_after_load = X.isna().sum().sort_values(ascending=False).to_frame("missing_count")

display(class_ratio)
display(missing_after_load)


In [ ]:
fig, axes = plt.subplots(len(CONTINUOUS_COLS), 2, figsize=(14, 4 * len(CONTINUOUS_COLS)))

for idx, col in enumerate(CONTINUOUS_COLS):
    sns.histplot(data=project_df, x=col, hue=TARGET_COL, kde=True, stat="density", common_norm=False, ax=axes[idx, 0])
    axes[idx, 0].set_title(f"{col} distribution by target")

    sns.boxplot(data=project_df, x=TARGET_COL, y=col, ax=axes[idx, 1])
    axes[idx, 1].set_title(f"{col} boxplot by target")

plt.tight_layout()
plt.show()


In [ ]:
iqr_rows = []

for col in CONTINUOUS_COLS:
    series = project_df[col].dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    mask = (project_df[col] < lower_bound) | (project_df[col] > upper_bound)
    outlier_count = int(mask.sum())

    iqr_rows.append(
        {
            "feature": col,
            "q1": round(q1, 3),
            "q3": round(q3, 3),
            "iqr": round(iqr, 3),
            "lower_bound": round(lower_bound, 3),
            "upper_bound": round(upper_bound, 3),
            "outlier_count": outlier_count,
            "outlier_ratio_pct": round(outlier_count / len(project_df) * 100, 2),
        }
    )

iqr_outlier_summary = pd.DataFrame(iqr_rows).sort_values("outlier_count", ascending=False)
display(iqr_outlier_summary)

plt.figure(figsize=(10, 4))
sns.barplot(data=iqr_outlier_summary, x="feature", y="outlier_count", color="tomato")
plt.title("IQR-based Outlier Count by Continuous Feature")
plt.xlabel("Feature")
plt.ylabel("Outlier Count")
plt.show()


In [ ]:
fig, axes = plt.subplots(len(CATEGORICAL_COLS), 1, figsize=(12, 3.2 * len(CATEGORICAL_COLS)))
if len(CATEGORICAL_COLS) == 1:
    axes = [axes]

for ax, col in zip(axes, CATEGORICAL_COLS):
    plot_df = project_df.groupby([col, TARGET_COL]).size().reset_index(name="count")
    sns.barplot(data=plot_df, x=col, y="count", hue=TARGET_COL, ax=ax)
    ax.set_title(f"{col} count by target")
    ax.set_xlabel(col)
    ax.set_ylabel("count")

plt.tight_layout()
plt.show()


In [ ]:
corr_df = project_df[CONTINUOUS_COLS + [TARGET_COL]].corr(numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(corr_df, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Continuous Features and Target Correlation")
plt.show()


## 3. 입력 검증과 전처리 파이프라인

`inference.py`는 임상 범위를 검사하고, `build_pipeline()`은 결측 대치와 스케일링/인코딩을 수행합니다. 테스트 코드에서 검증하는 조건도 여기서 같이 확인합니다.


In [ ]:
range_summary = pd.DataFrame(
    [
        {
            "feature": feature,
            "min_allowed": min_allowed,
            "max_allowed": max_allowed,
            "observed_min": X[feature].min(),
            "observed_max": X[feature].max(),
        }
        for feature, (min_allowed, max_allowed) in VALID_RANGES.items()
        if feature in X.columns
    ]
)

sample_input = pd.read_csv(DATA_DIR / "sample_input.csv")
training_range_violations = validate_input_ranges(X)
sample_range_violations = validate_input_ranges(sample_input[FEATURE_COLS])

display(range_summary)
print(f"Training data range violations: {training_range_violations}")
print(f"sample_input.csv range violations: {sample_range_violations}")


In [ ]:
pipeline = build_pipeline()
X_transformed = pipeline.fit_transform(X)
transformed_columns = pipeline.named_steps["preprocessor"].get_feature_names_out()
transformed_df = pd.DataFrame(X_transformed, columns=transformed_columns, index=X.index)

print(pipeline)
print(f"Input shape: {X.shape}")
print(f"Transformed shape: {X_transformed.shape}")
transformed_df.head()


In [ ]:
X_transformed_again = pipeline.transform(X)
second_pipeline = build_pipeline()
X_transformed_refit = second_pipeline.fit_transform(X)

assert X_transformed.shape == (len(X), len(FEATURE_COLS))
assert not np.isnan(X_transformed).any()
np.testing.assert_allclose(X_transformed, X_transformed_again)
np.testing.assert_allclose(X_transformed, X_transformed_refit)

print("Preprocessing checks passed:")
print("- output shape matches number of input features")
print("- no missing values remain after preprocessing")
print("- transform output is deterministic")


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_summary = pd.DataFrame(
    {
        "rows": [len(X_train), len(X_test)],
        "target_mean": [y_train.mean(), y_test.mean()],
    },
    index=["train", "test"],
)

display(split_summary)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train no disease: {(y_train == 0).sum()} | disease: {(y_train == 1).sum()}")
print(f"Test no disease: {(y_test == 0).sum()} | disease: {(y_test == 1).sum()}")


## 4. 다음 단계 정리

- `train.py`는 여기서 만든 전처리 결과 위에 `SelectFromModel`과 분류기를 붙여 학습합니다.
- `inference.py`는 `FEATURE_COLS` 순서와 `VALID_RANGES`를 그대로 사용해 입력을 검증합니다.
- `monitor.py`는 `CONTINUOUS_COLS`에 대해 KS 검정 기반 드리프트를 계산하고 성능 시계열을 저장합니다.
- 따라서 이 노트북의 핵심 산출물은 데이터 구조 이해, 결측/범위 확인, 전처리 검증입니다.
